In [24]:
# مكتبات تحليل البيانات
import pandas as pd
import numpy as np

# مكتبات معالجة البيانات
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# مكتبات النماذج في Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# مكتبات التعلم العميق (Keras)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input, SimpleRNN, GRU, LSTM
from tensorflow.keras.utils import to_categorical

In [2]:
data_f = pd.read_csv('UNSW_NB15_training-set.csv')

In [3]:
data_sample = data_f.sample(frac=0.9, random_state=42)
X = data_sample.drop(columns=['id', 'label', 'attack_cat'])
y = data_sample['label']

categorical_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [4]:
features = ['dur', 'sbytes', 'dbytes', 'rate', 'smean', 'dmean']

X = data_f[features]
y = data_f['label']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [6]:
ml_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(criterion='gini', max_depth=5,random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier()
}

In [7]:
results = []
for model_name, ml_model in ml_models.items():
    ml_model.fit(X_train, y_train)  # استخدام البيانات المعالجة مباشرة
    y_pred = ml_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results.append({"Model": model_name, "Accuracy": accuracy})

print("Machine Learning Model Evaluation Results:")
for result in results:
    print(f"{result['Model']}: {result['Accuracy']:.4f}")

Machine Learning Model Evaluation Results:
Logistic Regression: 0.7465
Decision Tree: 0.8307
K-Nearest Neighbors: 0.9225


In [25]:
# تحويل y إلى صيغة الفئات
y_categorical = to_categorical(y)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.3, random_state=42)

In [27]:
# تقييس البيانات
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
# إعداد البيانات لـ CNN
cnn_height, cnn_width = 5, 8
padding_needed = (cnn_height * cnn_width) - X_train_scaled.shape[1]
X_train_padded = np.hstack((X_train_scaled, np.zeros((X_train_scaled.shape[0], padding_needed))))
X_test_padded = np.hstack((X_test_scaled, np.zeros((X_test_scaled.shape[0], padding_needed))))

In [29]:
X_train_cnn = X_train_padded.reshape(-1, cnn_height, cnn_width, 1)
X_test_cnn = X_test_padded.reshape(-1, cnn_height, cnn_width, 1)

In [30]:
# إعداد البيانات لـ RNN, LSTM, GRU
total_elements_per_sample = cnn_height * cnn_width
X_train_rnn = X_train_padded.reshape(-1, total_elements_per_sample, 1)
X_test_rnn = X_test_padded.reshape(-1, total_elements_per_sample, 1)

In [31]:
# قائمة لحفظ النتائج
results = []

# نموذج CNN
cnn_model = Sequential([
    Input(shape=(cnn_height, cnn_width, 1)),
    Conv2D(32, kernel_size=(3, 3), activation='relu'),
    Flatten(),
    Dense(50, activation='relu'),
    Dense(y_train.shape[1], activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_split=0.2)
cnn_accuracy = cnn_model.evaluate(X_test_cnn, y_test)[1]
results.append({'Model': 'CNN', 'Accuracy': cnn_accuracy})

# نموذج SimpleRNN
rnn_model = Sequential([
    Input(shape=(total_elements_per_sample, 1)),
    SimpleRNN(50, activation='relu'),
    Dense(25, activation='relu'),
    Dense(y_train.shape[1], activation='softmax')
])
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
rnn_model.fit(X_train_rnn, y_train, epochs=10, batch_size=32, validation_split=0.2)
rnn_accuracy = rnn_model.evaluate(X_test_rnn, y_test)[1]
results.append({'Model': 'SimpleRNN', 'Accuracy': rnn_accuracy})

# نموذج LSTM
lstm_model = Sequential([
    Input(shape=(total_elements_per_sample, 1)),
    LSTM(50, activation='tanh'),
    Dense(25, activation='relu'),
    Dense(y_train.shape[1], activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_train_rnn, y_train, epochs=10, batch_size=32, validation_split=0.2)
lstm_accuracy = lstm_model.evaluate(X_test_rnn, y_test)[1]
results.append({'Model': 'LSTM', 'Accuracy': lstm_accuracy})

# نموذج GRU
gru_model = Sequential([
    Input(shape=(total_elements_per_sample, 1)),
    GRU(50, activation='tanh'),
    Dense(25, activation='relu'),
    Dense(y_train.shape[1], activation='softmax')
])
gru_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
gru_model.fit(X_train_rnn, y_train, epochs=10, batch_size=32, validation_split=0.2)
gru_accuracy = gru_model.evaluate(X_test_rnn, y_test)[1]
results.append({'Model': 'GRU', 'Accuracy': gru_accuracy})

# طباعة النتائج
print("Model Evaluation Results:")
for result in results:
    print(f"{result['Model']}: {result['Accuracy']:.4f}")

Epoch 1/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7463 - loss: 0.5255 - val_accuracy: 0.7962 - val_loss: 0.4189
Epoch 2/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7962 - loss: 0.4114 - val_accuracy: 0.8349 - val_loss: 0.3515
Epoch 3/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8327 - loss: 0.3481 - val_accuracy: 0.8553 - val_loss: 0.3181
Epoch 4/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8563 - loss: 0.3086 - val_accuracy: 0.8393 - val_loss: 0.3014
Epoch 5/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8636 - loss: 0.2914 - val_accuracy: 0.8726 - val_loss: 0.2777
Epoch 6/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8678 - loss: 0.2800 - val_accuracy: 0.8656 - val_loss: 0.2859
Epoch 7/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.8713 - loss: 0.2729 - val_accuracy: 0.8714 - val_loss: 0.2662
Epoch 8/10
1441/1441 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8756 - loss: 0.2677 - 